# **Diplomado IA: Aplicaciones 2 - Audio y Video**. <br> Práctico 44: Deep Fakes
---
---

**Profesor:**
- Carlos Aspillaga

**Ayudante:**
- Por definir
---
---

# **Instrucciones Generales**

El siguiente práctico se debe realizar **individualmente**. El formato de entrega es el **archivo .ipynb con todas las celdas ejecutadas**. Todas las preguntas deben ser respondidas en las celdas dispuestas para ello. No se aceptará el _output_ de una celda de código como respuesta.

**Nombre alumno:** Roberto Araneda

El siguiente práctico cuenta con varias secciones y en varias de ellas se incluye actividades a realizar.

**IMPORTANTE: habrá un bonus de 1 décima para todos aquellos alumnos/as que muestren buen orden en sus respuestas (esto aplica a legibilidad de código, buena redacción, formalidad, organización del jupyter notebook, seguimiento de instrucciones, etc). El criterio lo pondrá cada ayudante corrector. La nota máxima obtenible en el laboratorio es 7.0**

In [1]:
import os

In [2]:
import os, shutil, subprocess, json
from google.colab import files
from IPython.display import Audio, HTML, display
from base64 import b64encode

print("Sube generated.wav y mi_deepfake.mp4 (puedes seleccionar los dos a la vez):")
up = files.upload()

for n in up:
    dest = '/content/generated.wav' if n.lower().endswith('.wav') else '/content/video.mp4'
    shutil.move(n, dest)
    print(f"  {n}  ->  {dest}")

def dur(path):
    out = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration',
                          '-of','json', path], capture_output=True, text=True).stdout
    return float(json.loads(out)['format']['duration'])

print()
for p in ['/content/generated.wav', '/content/video.mp4']:
    print(f"{p:26} {os.path.getsize(p)/1e6:6.2f} MB   {dur(p):.2f} s")

print(f"\naudio + 0,7 s de silencio = {dur('/content/generated.wav')+0.7:.2f} s")
print(f"video                      = {dur('/content/video.mp4'):.2f} s")
print("OK: el video es mas largo, -shortest cortara limpio"
      if dur('/content/video.mp4') > dur('/content/generated.wav')+0.7
      else "⚠ el video es mas corto: habria que estirarlo con setpts (celda 5)")

display(Audio('/content/generated.wav'))
display(HTML(f'<video width=520 controls loop><source src="data:video/mp4;base64,'
             f'{b64encode(open("/content/video.mp4","rb").read()).decode()}"></video>'))


Sube generated.wav y mi_deepfake.mp4 (puedes seleccionar los dos a la vez):


Saving generated.wav to generated.wav
Saving mi_deepfake.mp4 to mi_deepfake.mp4
  generated.wav  ->  /content/generated.wav
  mi_deepfake.mp4  ->  /content/video.mp4

/content/generated.wav       0.34 MB   7.05 s
/content/video.mp4           2.38 MB   8.00 s

audio + 0,7 s de silencio = 7.75 s
video                      = 8.00 s
OK: el video es mas largo, -shortest cortara limpio


# **Funcionalidades para acelerar el frame-rate**
(en caso de ser necesario)

In [3]:
import shutil
shutil.copy('/content/video.mp4', '/content/input_video.mp4')
print("listo: /content/input_video.mp4 creado a partir del deepfake")


listo: /content/input_video.mp4 creado a partir del deepfake


In [4]:
# Aumentar frame rate y acelerar 3x, para restablecer la velocidad original
!ffmpeg -y -loglevel error -stats -i /content/input_video.mp4 -vcodec libx264 -acodec aac -filter:v "fps=90" /content/input_video2.mp4
!ffmpeg -y -loglevel error -stats -i /content/input_video2.mp4 -vcodec libx264 -acodec aac -filter:v "setpts=0.33333*PTS" /content/input_video3.mp4
!ffmpeg -y -loglevel error -stats -i /content/input_video2.mp4 -vcodec libx264 -acodec aac -filter:v "setpts=2.0*PTS" /content/input_video4.mp4

frame=  720 fps= 93 q=-1.0 Lsize=     408kB time=00:00:07.96 bitrate= 419.5kbits/s speed=1.03x    
frame=  242 fps= 73 q=-1.0 Lsize=     200kB time=00:00:02.65 bitrate= 617.5kbits/s dup=0 drop=478 speed=0.803x    
frame= 1439 fps=119 q=-1.0 Lsize=     551kB time=00:00:15.95 bitrate= 283.0kbits/s dup=719 drop=0 speed=1.32x    


## **Funcionalidad para visualizar un video en Google Colab**

In [5]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('/content/input_video3.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=256 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

# **Funcionalidad para agregar silencio al inicio o al final de un audio**
(en caso de ser necesario)

In [6]:
# agregar 0.7s de silencio al inicio
! ffmpeg -y -loglevel error -stats -f lavfi -t 0.7 -i anullsrc=channel_layout=stereo:sample_rate=44100 -i /content/generated.wav -filter_complex "[0:a][1:a]concat=n=2:v=0:a=1" /content/generated2.wav

# rellenar con silencio al final (de tal manera que el audio tenga duración >= que el video, pues sino el FFMPEG dejaría el video del tamaño del más corto)
!ffmpeg -y -loglevel error -stats -f lavfi -t 10 -i anullsrc=channel_layout=stereo:sample_rate=44100 -i /content/generated2.wav -filter_complex "[1:a][0:a]concat=n=2:v=0:a=1" /content/generated2.wav


size=     363kB time=00:00:07.69 bitrate= 386.7kbits/s speed=1.67e+03x    
Output /content/generated2.wav same as Input #1 - exiting


In [7]:
!ffmpeg -y -loglevel error -i /content/generated.wav -af "adelay=700:all=1" /content/audio_adelay.wav

import subprocess, json
def info(p):
    o = subprocess.run(['ffprobe','-v','error','-select_streams','a:0','-show_entries',
        'stream=sample_rate,channels,bits_per_raw_sample:format=duration','-of','json',p],
        capture_output=True, text=True).stdout
    j = json.loads(o); s = j['streams'][0]
    return float(j['format']['duration']), s['sample_rate'], s['channels']

for p in ['/content/generated.wav', '/content/generated2.wav', '/content/audio_adelay.wav']:
    try:
        d, sr, ch = info(p)
        print(f"{p:32} {d:5.2f} s   {sr} Hz   {ch} ch")
    except Exception as e:
        print(f"{p:32} ILEGIBLE ({type(e).__name__})")

from IPython.display import Audio, display
for p in ['/content/generated2.wav', '/content/audio_adelay.wav']:
    print(p); display(Audio(p))


/content/generated.wav            7.05 s   24000 Hz   1 ch
/content/generated2.wav           7.75 s   24000 Hz   1 ch
/content/audio_adelay.wav         7.75 s   24000 Hz   1 ch
/content/generated2.wav


/content/audio_adelay.wav


In [8]:
import shutil
shutil.copy('/content/generated2.wav', '/content/audio.wav')   # o audio_adelay.wav
print("audio.wav listo, es lo que lee la celda 11")

audio.wav listo, es lo que lee la celda 11


# **Funcionalidad para unir audio y video**

In [10]:

# juntar audio (la pista con 0.7s de silencio) y el video generado por Deep Fake
!ffmpeg -y -loglevel error -stats -i /content/video.mp4 -i /content/audio.wav -c:v copy -c:a aac -shortest /content/final.mp4

frame=  127 fps=0.0 q=-1.0 Lsize=    2382kB time=00:00:07.75 bitrate=2517.3kbits/s speed= 124x    


In [11]:
import subprocess, json
from IPython.display import HTML, display
from base64 import b64encode

o = subprocess.run(['ffprobe','-v','error','-show_entries',
     'stream=index,codec_type,codec_name,width,height,r_frame_rate,sample_rate,channels:format=duration',
     '-of','json','/content/final.mp4'], capture_output=True, text=True).stdout
j = json.loads(o)
print(f"duracion: {float(j['format']['duration']):.2f} s")
for s in j['streams']:
    print(" ", {k: v for k, v in s.items() if v is not None})

display(HTML(f'<video width=640 controls><source src="data:video/mp4;base64,'
             f'{b64encode(open("/content/final.mp4","rb").read()).decode()}" type="video/mp4"></video>'))

from google.colab import files
files.download('/content/final.mp4')


duracion: 8.00 s
  {'index': 0, 'codec_name': 'h264', 'codec_type': 'video', 'width': 832, 'height': 464, 'r_frame_rate': '16/1'}
  {'index': 1, 'codec_name': 'aac', 'codec_type': 'audio', 'sample_rate': '24000', 'channels': 1, 'r_frame_rate': '0/0'}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Actividad**

Para esta actividad usted tendrá que generar un DeepFake a partir de un video generado por usted, diferente al mostrado en clases.

**Requisitos:**

**Generación de audio:**
* La frase debe ser diferente a la utilizada en clases
* Es altamente recomendado utilizar una voz diferente a la de Barack Obama, pero está permitido utilizar el mismo archivo de voz de Barack Obama (discurso de youtube) usado en clases, siempre y cuando respete los otros requisitos aquí mencionados. No habrá penalización si opta por usar este mismo audio como referencia de voz.

**Generación de video:**
* Debe generar un video propio, generado de acuerdo con el texto propuesto por usted. En caso de dificultad, puede buscar un video de referencia de internet, pero posiblemente se le penalice si no consigue alinear el video y la voz generada.
* Debe utilizar una imagen de referencia diferente a la entregada en clases. No hay problema si usa a Barack Obama, mientras la fotografía sea diferente.

**Requisitos y sugerencias generales:**
* Una vez que genere un audio que le guste, ¡guárdelo a su computador! Porque si Colab se reinicia, perderá ese resultado. En tal caso puede volver a subir ese archivo a la nueva sesion de colab y continuar desde ahi.
* No está permitido generar material ofensivo, inmoral, que contenga improperios o insultos. De ser así, el ayudante podrá poner un 1.0 a su entrega sin derecho a reclamo. ¡No se arriesgue! Busque algo entretenido y sano, adecuado para una entrega en la Universidad Católica.
* Es altamente recomendado participar de la clase y seguir los tips entregados por el profesor. No todo es tan sencillo como parece, así que la guía de la clase puede ahorrarle varios dolores de cabeza y posiblemente le permita lograr un mejor resultado.
* En caso que no logre terminar el labortorio, se aceptará entregas parciales. Cada etapa tendrá su propio puntaje, así que es mejor entregar incompleto que no entregar nada.
* Sí se aceptará videos donde el deep Fake falla (el modelo no logra recrear correctamente el movimiento) con penalización. Por lo mismo se sugiere seguir las recomendaciones de la clase para lograr un buen resultado.
* No habrá penalización por audios defectuosos (sonido robótico) pues es algo propio del modelo. De todas formas se sugiere correr la celda varias veces hasta conseguir un resultado de su agrado.
* IMPORTANTE: Debe entregar los 3 jupyter notebooks con todas las celdas ejecutadas y el output disponible para verse. Si no hace esto, el ayudante no podrá corregir su entrega y obtendrá puntaje cero en ese ítem. En caso de problemas, puede adjuntar el video y el audio generados en el buzón de entrega, junto con el jupyter notebook.


## Resumen del flujo

| Etapa | Herramienta | Salida | Duración |
|---|---|---|---|
| Clonación de voz | Tortoise-TTS (`fast`, k=3) | `generated.wav` | 7,05 s |
| Animación | Wan2.2-Animate-14B GGUF Q4_K_M | `mi_deepfake.mp4` | 8,00 s |
| Silencio inicial | ffmpeg (`anullsrc` + `concat`) | `audio.wav` | 7,75 s |
| Unión | ffmpeg (`-c:v copy -c:a aac -shortest`) | `final.mp4` | 7,75 s |

**Insumos.** Frase propia en inglés, voz de referencia del discurso de graduación de
Barack Obama (90 s descargados, de los cuales el modelo usa 18 s en la rama
autoregresiva y 12,8 s en la de difusión). Video de movimiento propio, grabado
haciendo playback del audio ya generado, con 0,7 s de margen antes y después.
Imagen de referencia: retrato oficial de 2012 (Pete Souza, Casa Blanca, dominio
público), recortada a 16:9 para coincidir con el encuadre del video y con el aspecto
de entrenamiento del modelo.

**Alineación.** Grabar en playback sobre el audio ya generado hace que la sincronía
labial sea exacta en lugar de aproximada. El número de frames se calculó desde la
duración medida del audio (7,05 s × 16 fps ≈ 113, con margen a 128), de modo que no
fue necesario usar las utilidades de `setpts` para estirar o acelerar el video.


## Observaciones

**Flujo.** Tortoise-TTS (`fast`, k=3) → `generated.wav` (7,05 s) · Wan2.2-Animate-14B
GGUF Q4_K_M (`t4-max`, 128 frames a 16 fps) → `mi_deepfake.mp4` (8,00 s) · ffmpeg →
`final.mp4` (7,75 s de audio sobre 8,00 s de video).

**Alineación.** El video de control se grabó haciendo *playback* del audio ya generado,
con 0,7 s de margen antes y después. La sincronía labial resulta exacta en lugar de
aproximada, y el margen coincide con el silencio que agrega la celda de la utilidad de
audio. `MAX_FRAMES` se calculó desde la duración medida del audio (no estimada: con la
misma frase, Tortoise produjo 6,12 / 7,05 / 6,87 s en tres candidatos), por lo que no
fue necesario usar las utilidades de `setpts`.

**Bugs de ffmpeg encontrados.**
1. La celda de silencio usa `generated2.wav` como entrada y salida del mismo comando.
   ffmpeg lo detecta y aborta (`Output ... same as Input #1 - exiting`). El archivo no se
   corrompe, pero el relleno final no se aplica. En este caso conviene: con el audio en
   7,75 s y el video en 8,00 s, el corte descarta el tramo final, que es el más degradado.
2. Ese mismo comando concatena `anullsrc` (estéreo, 44.100 Hz) con audio mono a
   24.000 Hz, formatos que `concat` exige uniformes. ffmpeg inserta un `aresample`
   implícito y preserva el formato original, como confirma el bitrate reportado:
   386,7 kbit/s ≈ 24.000 Hz × 16 bits × 1 canal.
3. `-shortest` no cortó a 7,75 s. En modo `-c:v copy` ffmpeg encola paquetes hacia el
   muxer sin decodificar, y el encoder AAC agrega padding (frames de 1024 muestras:
   7,75 s × 24.000 = 186.000, redondeado a 182 × 1024 = 186.368). Se verificó con un
   recorte explícito `-t 7.75 -c copy`.
4. La utilidad de frame rate lee `/content/input_video.mp4`, archivo que ningún notebook
   del laboratorio crea; proviene del flujo original con FOMM. Además `setpts` se aplica
   como `filter:v`: sobre un video con audio lo desincronizaría, y habría que encadenar
   `atempo` (limitado a [0,5 , 2,0] por invocación).

**Limitación del resultado.** El parecido con la imagen de referencia se degrada.
Concurren tres factores: la imagen (retrato oficial de 2012) tiene una sonrisa amplia
mientras el video de control tiene expresión seria, de modo que el modelo debe
transformar toda la mitad inferior del rostro; Q4_K_M cuantiza también el adaptador
facial; y a 832×464 el rostro ocupa unos 180×230 px. El paper de Wan-Animate advierte
justamente sobre la pérdida de consistencia de identidad en escenarios *cross-identity*
con disparidad de forma facial. El arreglo de menor costo sería una imagen de referencia
con expresión neutra, que elimina el desajuste sin regrabar el video.

**Lo que puso el prior.** La imagen de referencia era un recorte de cabeza y hombros,
sin brazos. El modelo generó brazos cruzados y extendió el fondo más allá del recorte, y
los artefactos se concentran precisamente ahí: en la región donde el esqueleto de control
no aportaba ninguna señal y todo lo aportó el prior aprendido.
